# Chapter 2 — Tokens and Token Embeddings
### Practice Notebook

*Source: Hands-On Large Language Models, Jay Alammar & Maarten Grootendorst (O'Reilly)*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HandsOnLLM/Hands-On-Large-Language-Models/blob/main/chapter02/Chapter%202%20-%20Tokens%20and%20Token%20Embeddings.ipynb)

---

This is your **practice workspace** for Chapter 2. Every code cell is a stub. The chapter covers the full journey from raw text to embeddings: tokenisation → token IDs → token embeddings → contextualized embeddings → sentence embeddings → word2vec → recommendation systems.

---

## Table of Contents

- [Part 1: Tokenisation Basics](#part-1-tokenisation-basics)
  - [Exercise 1.1 — Tokenise a Prompt and Inspect Token IDs](#exercise-11--tokenise-a-prompt-and-inspect-token-ids)
  - [Exercise 1.2 — Decode Individual Tokens](#exercise-12--decode-individual-tokens)
  - [Exercise 1.3 — Generate Text and Inspect Output Tokens](#exercise-13--generate-text-and-inspect-output-tokens)
  - [Exercise 1.4 — Token Count vs Character Count](#exercise-14--token-count-vs-character-count)
- [Part 2: Comparing Tokenizers](#part-2-comparing-tokenizers)
  - [Exercise 2.1 — Build the show_tokens Visualiser](#exercise-21--build-the-show_tokens-visualiser)
  - [Exercise 2.2 — Compare Tokenizers on English Text](#exercise-22--compare-tokenizers-on-english-text)
  - [Exercise 2.3 — Compare Tokenizers on Code and Emojis](#exercise-23--compare-tokenizers-on-code-and-emojis)
  - [Exercise 2.4 — Tokenizer Properties Comparison Table](#exercise-24--tokenizer-properties-comparison-table)
- [Part 3: From Tokens to Contextualized Embeddings](#part-3-from-tokens-to-contextualized-embeddings)
  - [Exercise 3.1 — Extract Contextualized Token Embeddings](#exercise-31--extract-contextualized-token-embeddings)
  - [Exercise 3.2 — Same Word, Different Context](#exercise-32--same-word-different-context)
- [Part 4: Sentence Embeddings](#part-4-sentence-embeddings)
  - [Exercise 4.1 — Encode Sentences](#exercise-41--encode-sentences)
  - [Exercise 4.2 — Semantic Similarity Ranking](#exercise-42--semantic-similarity-ranking)
- [Part 5: Traditional Word Embeddings (GloVe)](#part-5-traditional-word-embeddings-glove)
  - [Exercise 5.1 — Load GloVe and Find Similar Words](#exercise-51--load-glove-and-find-similar-words)
  - [Exercise 5.2 — Word Vector Arithmetic](#exercise-52--word-vector-arithmetic)
- [Part 6: Word2Vec for Music Recommendation](#part-6-word2vec-for-music-recommendation)
  - [Exercise 6.1 — Load Playlist Data](#exercise-61--load-playlist-data)
  - [Exercise 6.2 — Train a Word2Vec Model on Playlists](#exercise-62--train-a-word2vec-model-on-playlists)
  - [Exercise 6.3 — Recommend Similar Songs](#exercise-63--recommend-similar-songs)

### [OPTIONAL] — Install packages on Colab

💡 **GPU required** for Part 1 and Part 3. Go to Runtime → Change runtime type → GPU (T4).

In [ ]:
# %%capture
# !pip install transformers sentence-transformers gensim scikit-learn accelerate peft scipy numpy

---
# Part 1: Tokenisation Basics

Language models do not read text — they read **token IDs**. A tokenizer converts raw text into a sequence of integers, each pointing to an entry in the model's vocabulary table. Understanding this pipeline is fundamental to understanding why LLMs behave the way they do.

```
Raw text  →  [Tokenizer]  →  Token IDs  →  [Model]  →  Output token IDs  →  [Tokenizer]  →  Text
```

## Exercise 1.1 — Tokenise a Prompt and Inspect Token IDs

**Task:**
1. Load the Phi-3-mini model and tokenizer
2. Tokenise this prompt: `"Write an email apologizing to Sarah for the tragic gardening mishap. Explain how it happened.<|assistant|>"`
3. Move the token IDs to CUDA
4. Print the raw tensor of token IDs
5. Print how many tokens the prompt contains

**Hint:** `tokenizer(prompt, return_tensors="pt").input_ids`

**Observation:** Count tokens vs characters — tokens are typically 3–4 characters on average for English text.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# YOUR CODE HERE
# Load model and tokenizer
# model = AutoModelForCausalLM.from_pretrained(
#     "microsoft/Phi-3-mini-4k-instruct",
#     device_map="cuda",
#     torch_dtype="auto",
#     trust_remote_code=True,
# )
# tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")

prompt = "Write an email apologizing to Sarah for the tragic gardening mishap. Explain how it happened.<|assistant|>"

# YOUR CODE HERE
# Tokenise and move to GPU
# input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to("cuda")

# print("Token IDs tensor:", input_ids)
# print(f"Number of tokens: {input_ids.shape[1]}")
# print(f"Number of characters: {len(prompt)}")
# print(f"Average chars per token: {len(prompt) / input_ids.shape[1]:.1f}")

## Exercise 1.2 — Decode Individual Tokens

Token IDs are just integers — decoding each one separately reveals how the tokenizer split the text into subword pieces.

**Task:** Loop through each token ID in `input_ids[0]` and decode it individually using `tokenizer.decode(id)`. Print each decoded token on its own line.

**Expected pattern:** Some tokens will be full words (`Write`, `an`, `email`), some will be subword fragments (`trag`, `ic`, `garden`, `ing`, `mishap`).

**Key insight:** This shows exactly how the tokenizer decided to split the text.

In [ ]:
print("Individual tokens:")
# YOUR CODE HERE
# for token_id in input_ids[0]:
#     print(repr(tokenizer.decode(token_id)))

## Exercise 1.3 — Generate Text and Inspect Output Tokens

**Task:**
1. Generate up to 20 new tokens from the prompt using `model.generate(input_ids=input_ids, max_new_tokens=20)`
2. Print the full output token IDs tensor
3. Decode the full output as a complete string using `tokenizer.decode(generation_output[0])`
4. Decode only the **new** generated tokens (from position `input_ids.shape[1]` onwards)

**Observation:** The output tensor includes both the input tokens and the newly generated ones.

In [ ]:
# YOUR CODE HERE
# generation_output = model.generate(input_ids=input_ids, max_new_tokens=20)

# print("Full output token IDs:")
# print(generation_output)

# print("\nFull decoded output:")
# print(tokenizer.decode(generation_output[0]))

# print("\nOnly the newly generated part:")
# new_tokens = generation_output[0][input_ids.shape[1]:]
# print(tokenizer.decode(new_tokens))

## Exercise 1.4 — Token Count vs Character Count

Different text types tokenise very differently. Understanding this helps you predict context window usage.

**Task:** For each sentence below, compute:
- Number of tokens
- Number of characters
- Characters per token ratio

**Sentences to test:**
```python
sentences = [
    "Hello world",                               # simple English
    "Pneumonoultramicroscopicsilicovolcanoconiosis",  # long rare word
    "for i in range(10): print(i)",              # Python code
    "😀🎉🚀💡",                                    # emojis
    "The quick brown fox jumps over the lazy dog",  # pangram
]
```

**Observation:** Code and rare words often use more tokens per character. Emojis can be many tokens each.

In [ ]:
sentences = [
    "Hello world",
    "Pneumonoultramicroscopicsilicovolcanoconiosis",
    "for i in range(10): print(i)",
    "😀🎉🚀💡",
    "The quick brown fox jumps over the lazy dog",
]

# YOUR CODE HERE
# For each sentence, tokenise with the Phi-3 tokenizer and print:
#   sentence[:30] | num_tokens | num_chars | ratio

# print(f"{'Sentence':<35} {'Tokens':>8} {'Chars':>8} {'Chars/Token':>12}")
# print("-" * 70)
# for s in sentences:
#     ...

---
# Part 2: Comparing Tokenizers

Different models use different tokenizers — different algorithms (BPE vs WordPiece), different vocabulary sizes (30K to 100K+), and different training data. The same input text can produce very different token sequences depending on the tokenizer.

| Tokenizer | Model | Algorithm | Vocab size |
|---|---|---|---|
| `bert-base-uncased` | BERT (2018) | WordPiece | 30,522 |
| `gpt2` | GPT-2 (2019) | BPE | 50,257 |
| `Xenova/gpt-4` | GPT-4 (2023) | BPE | ~100,000 |
| `microsoft/Phi-3-mini-4k-instruct` | Phi-3 (2024) | BPE | 32,000 |

## Exercise 2.1 — Build the `show_tokens` Visualiser

The book uses a `show_tokens` function that prints each token in a different colour, making tokenizer behaviour immediately visible.

**Task:** Implement `show_tokens(sentence, tokenizer_name)` that:
1. Loads the tokenizer by name
2. Tokenises the sentence
3. Prints each decoded token with alternating ANSI background colours

**ANSI colour format:** `\x1b[48;2;R;G;Bm` sets background to RGB. Use `\x1b[0m` to reset.

**Colour list to use:**
```python
colors = [
    '48;2;102;194;165',  # teal
    '48;2;252;141;98',   # orange
    '48;2;141;160;203',  # blue
    '48;2;231;138;195',  # pink
    '48;2;166;216;84',   # green
    '48;2;255;217;47',   # yellow
]
```

In [ ]:
from transformers import AutoTokenizer

colors = [
    '48;2;102;194;165',
    '48;2;252;141;98',
    '48;2;141;160;203',
    '48;2;231;138;195',
    '48;2;166;216;84',
    '48;2;255;217;47',
]


def show_tokens(sentence: str, tokenizer_name: str) -> None:
    """Print each token of sentence in alternating colours using the given tokenizer."""
    # YOUR CODE HERE
    # 1. Load tokenizer from tokenizer_name
    # 2. Tokenise sentence to get token_ids
    # 3. For each (idx, token_id), decode and print with colors[idx % len(colors)]
    # Format: print(f'\x1b[{colors[idx % len(colors)]}m' + decoded_token + '\x1b[0m', end='')
    # 4. Print a newline at the end
    pass


# Quick test
show_tokens("Hello world, how are you?", "bert-base-uncased")

## Exercise 2.2 — Compare Tokenizers on English Text

**Task:** Run `show_tokens` on the sentence below with all four tokenizers. After each, print the total token count.

**Sentence:** `"Tokenization is the first step in understanding large language models. CAPITALIZATION matters!"`

**Tokenizers to compare:**
- `"bert-base-uncased"` — WordPiece, lowercases everything, `##` prefix for continuation tokens
- `"bert-base-cased"` — WordPiece, preserves case
- `"gpt2"` — BPE, preserves case and whitespace
- `"microsoft/Phi-3-mini-4k-instruct"` — BPE, modern vocabulary

**Observe:** How does BERT handle the capital letters? How does GPT-2 handle the leading space before words?

In [ ]:
sentence = "Tokenization is the first step in understanding large language models. CAPITALIZATION matters!"

tokenizer_names = [
    "bert-base-uncased",
    "bert-base-cased",
    "gpt2",
    "microsoft/Phi-3-mini-4k-instruct",
]

# YOUR CODE HERE
# For each tokenizer_name:
#   print the tokenizer name
#   call show_tokens(sentence, tokenizer_name)
#   load the tokenizer and print total token count
#   print a blank line between tokenizers

# for name in tokenizer_names:
#     print(f"\n{name}:")
#     ...

## Exercise 2.3 — Compare Tokenizers on Code and Emojis

Tokenizers trained on code handle indentation and symbols very differently from text-focused tokenizers. Emojis often require many tokens each.

**Task:** Run `show_tokens` with `bert-base-uncased` and `gpt2` on these two inputs:

**Code snippet:**
```python
code = "for i in range(10):\n    print(i)"
```

**Emoji string:**
```python
emojis = "I love 🤖 and 🎉"
```

**Observe:**
- How does BERT handle newlines and indentation?
- How many tokens does each emoji become in GPT-2?
- What does BERT do with unknown characters like emojis?

In [ ]:
code = "for i in range(10):\n    print(i)"
emojis = "I love 🤖 and 🎉"

# YOUR CODE HERE
# Show both inputs with bert-base-uncased and gpt2
# Print token counts too

# for text_label, text in [("Code", code), ("Emojis", emojis)]:
#     for tokenizer_name in ["bert-base-uncased", "gpt2"]:
#         ...

## Exercise 2.4 — Tokenizer Properties Comparison Table

**Task:** Fill in this table by loading each tokenizer and inspecting its vocabulary size and special tokens.

For each tokenizer, print:
- `tokenizer.vocab_size`
- `tokenizer.all_special_tokens`

**Tokenizers:** `bert-base-uncased`, `gpt2`, `microsoft/Phi-3-mini-4k-instruct`

In [ ]:
tokenizers_to_inspect = [
    "bert-base-uncased",
    "gpt2",
    "microsoft/Phi-3-mini-4k-instruct",
]

# YOUR CODE HERE
# For each tokenizer name:
#   load with AutoTokenizer.from_pretrained
#   print: name, vocab_size, all_special_tokens

# for name in tokenizers_to_inspect:
#     tok = AutoTokenizer.from_pretrained(name)
#     print(f"{name}")
#     print(f"  Vocab size:     {tok.vocab_size}")
#     print(f"  Special tokens: {tok.all_special_tokens}")
#     print()

---
# Part 3: From Tokens to Contextualized Embeddings

Every token in a vocabulary has a **static token embedding** — a vector that is the same regardless of context. But language models produce **contextualized embeddings** — the vector for a token changes depending on the surrounding tokens.

The word *"bank"* has a different contextualized embedding in *"river bank"* vs *"bank account"* — same token ID, completely different output vector.

```
Static:        "bank" → always [0.23, -0.11, 0.87, ...]

Contextualized: "bank" in "river bank" → [0.71, 0.34, -0.22, ...]   (water context)
                "bank" in "bank account" → [-0.18, 0.92, 0.43, ...]  (finance context)
```

## Exercise 3.1 — Extract Contextualized Token Embeddings

We use **DeBERTa** (an improved BERT) as the encoder to extract contextualized embeddings.

**Task:**
1. Load `microsoft/deberta-v3-xsmall` model and `microsoft/deberta-base` tokenizer
2. Tokenise `"Hello world"`
3. Pass through the model: `output = model(**tokens)[0]`
4. Print the output shape — expected `torch.Size([1, 4, 384])`
5. Print what the 4 tokens are (hint: decode each token ID)
6. Print the embedding vector for the first real token (index 1, skipping [CLS])

**Shape explained:** `[batch_size=1, num_tokens=4, embedding_dim=384]`

*Note: The 4 tokens are `[CLS]`, `Hello`, `world`, `[SEP]` — BERT-style models wrap input with special tokens.*

In [ ]:
from transformers import AutoModel, AutoTokenizer

# YOUR CODE HERE
# Load tokenizer and model
# tokenizer_deberta = AutoTokenizer.from_pretrained("microsoft/deberta-base")
# model_deberta = AutoModel.from_pretrained("microsoft/deberta-v3-xsmall")

# Tokenise
# tokens = tokenizer_deberta("Hello world", return_tensors="pt")

# Forward pass
# output = model_deberta(**tokens)[0]

# Print shape
# print("Output shape:", output.shape)
# print("[batch_size, num_tokens, embedding_dim]")

# Print the 4 tokens
# print("\nTokens:")
# for token_id in tokens['input_ids'][0]:
#     print(f"  {tokenizer_deberta.decode(token_id)}")

# Print embedding for 'Hello' (index 1)
# print("\nContextualized embedding for 'Hello' (first 10 dims):")
# print(output[0][1][:10])

## Exercise 3.2 — Same Word, Different Context

This exercise proves that contextualized embeddings actually differ based on context.

**Task:**
1. Tokenise both sentences: `"I went to the river bank"` and `"I deposited money at the bank"`
2. Find the index of the token `"bank"` in each sentence
3. Extract the embedding for `"bank"` from each sentence
4. Compute cosine similarity between the two `"bank"` embeddings

**Expected:** Similarity should be noticeably less than 1.0 — the same word has different representations in different contexts.

**Hint for finding the token index:**
```python
token_ids = tokenizer_deberta(sentence, return_tensors="pt")['input_ids'][0]
bank_id = tokenizer_deberta.convert_tokens_to_ids('bank')
bank_idx = (token_ids == bank_id).nonzero()[0].item()
```

In [ ]:
import torch
import torch.nn.functional as F

sentence_a = "I went to the river bank"
sentence_b = "I deposited money at the bank"

# YOUR CODE HERE
# 1. Tokenise both sentences
# 2. Run through model to get contextualized embeddings
# 3. Find the index of 'bank' in each token sequence
# 4. Extract the 'bank' embedding from each sentence
# 5. Compute cosine similarity between the two embeddings

# print(f"Cosine similarity of 'bank' in two contexts: {similarity:.4f}")
# print("If < 1.0, the embeddings differ by context — that is contextualization!")

---
# Part 4: Sentence Embeddings

Contextualized embeddings give one vector *per token*. For tasks like semantic search or document clustering, you need one vector *per sentence*. **Sentence embedding models** (from `sentence-transformers`) are specifically trained to produce a single vector that captures the full meaning of a sentence.

## Exercise 4.1 — Encode Sentences

**Task:**
1. Load `sentence-transformers/all-mpnet-base-v2`
2. Encode `"Best movie ever!"`
3. Print the vector shape — expected `(768,)`
4. Print the first 10 values of the vector

*Note: `all-mpnet-base-v2` produces 768-dim vectors (vs `all-MiniLM-L6-v2` which produces 384-dim).*

In [ ]:
from sentence_transformers import SentenceTransformer

# YOUR CODE HERE
# st_model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')
# vector = st_model.encode("Best movie ever!")

# print(f"Embedding shape: {vector.shape}")
# print(f"First 10 dimensions: {vector[:10]}")

## Exercise 4.2 — Semantic Similarity Ranking

**Task:** Given a query sentence, rank all candidates by semantic similarity to the query.

**Query:** `"The film was absolutely wonderful"`

**Candidates:**
```python
candidates = [
    "I hated every minute of this movie",
    "What a great picture, loved it!",
    "The weather today is sunny and warm",
    "An incredible cinematic experience",
    "The acting was mediocre at best",
]
```

Steps:
1. Encode the query and all candidates
2. Compute cosine similarity between the query and each candidate
3. Sort candidates from most similar to least similar
4. Print the ranked list with similarity scores

**Expected:** *"What a great picture"* and *"An incredible cinematic experience"* should rank highest.

In [ ]:
import numpy as np

query = "The film was absolutely wonderful"
candidates = [
    "I hated every minute of this movie",
    "What a great picture, loved it!",
    "The weather today is sunny and warm",
    "An incredible cinematic experience",
    "The acting was mediocre at best",
]

# YOUR CODE HERE
# 1. Encode query and all candidates
# 2. Compute cosine similarity for each candidate
# 3. Sort by similarity score (descending)
# 4. Print ranked results

# query_emb = st_model.encode(query)
# candidate_embs = st_model.encode(candidates)

# def cosine(a, b):
#     return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# scores = [(cosine(query_emb, c_emb), c) for c_emb, c in zip(candidate_embs, candidates)]
# scores.sort(reverse=True)

# print(f"Query: '{query}'\n")
# print("Ranked candidates (most similar first):")
# for score, candidate in scores:
#     print(f"  {score:.4f}  |  {candidate}")

---
# Part 5: Traditional Word Embeddings (GloVe)

Before transformer-based models, **GloVe** and **Word2Vec** were the dominant word embedding methods. They produce a single static vector per word (no context). They are faster and smaller than LLMs, and still useful for understanding the word2vec training intuition that underlies modern embeddings.

GloVe is pre-trained on Wikipedia — you download it directly via `gensim.downloader`.

## Exercise 5.1 — Load GloVe and Find Similar Words

**Task:**
1. Load the `glove-wiki-gigaword-50` model via `gensim.downloader` (~66MB)
2. Find the 10 most similar words to `"king"` using `model.most_similar`
3. Find the 10 most similar words to `"machine"`
4. Print both lists with similarity scores

**Expected for 'king':** queen, prince, emperor, throne, ... (royalty cluster)

In [ ]:
import gensim.downloader as api

# YOUR CODE HERE
# glove = api.load("glove-wiki-gigaword-50")
# print(f"Vocabulary size: {len(glove.key_to_index)}")
# print(f"Embedding dimension: {glove.vector_size}")

# print("\nMost similar to 'king':")
# for word, score in glove.most_similar('king', topn=10):
#     print(f"  {score:.4f}  {word}")

# print("\nMost similar to 'machine':")
# for word, score in glove.most_similar('machine', topn=10):
#     print(f"  {score:.4f}  {word}")

## Exercise 5.2 — Word Vector Arithmetic

One of the most famous properties of word embeddings is that semantic relationships can be expressed as vector arithmetic:

$$\vec{\text{king}} - \vec{\text{man}} + \vec{\text{woman}} \approx \vec{\text{queen}}$$

The idea: subtract the *man-ness* from *king*, add *woman-ness*, and you get something close to *queen*.

**Task:** Implement the following analogies using `model.most_similar(positive=[...], negative=[...])`:

1. `king - man + woman = ?` → expected: queen
2. `Paris - France + Italy = ?` → expected: Rome
3. `bigger - big + small = ?` → expected: smaller

**Hint:** `glove.most_similar(positive=['king', 'woman'], negative=['man'], topn=3)`

In [ ]:
analogies = [
    {
        "description": "king - man + woman",
        "positive": ["king", "woman"],
        "negative": ["man"],
        "expected": "queen"
    },
    {
        "description": "Paris - France + Italy",
        "positive": ["paris", "italy"],
        "negative": ["france"],
        "expected": "rome"
    },
    {
        "description": "bigger - big + small",
        "positive": ["bigger", "small"],
        "negative": ["big"],
        "expected": "smaller"
    },
]

# YOUR CODE HERE
# For each analogy, call glove.most_similar and print top 3 results
# Indicate whether the expected answer appeared in the top 3

# for analogy in analogies:
#     results = glove.most_similar(positive=analogy['positive'], negative=analogy['negative'], topn=3)
#     ...

---
# Part 6: Word2Vec for Music Recommendation

The book's most creative example: training Word2Vec on **playlists** instead of sentences. Each playlist is a sequence of song IDs — just like a sentence is a sequence of word IDs. Songs that often appear together in the same playlists end up with similar embeddings, enabling music recommendation by nearest-neighbour lookup.

```
Sentence:  ["the", "quick", "brown", "fox"] → words that co-occur get similar embeddings
Playlist:  ["2172", "1543", "889", "3021"] → songs that co-occur get similar embeddings
```

The insight: **anything sequential can be embedded this way** — songs, products, user actions, API calls.

## Exercise 6.1 — Load Playlist Data

**Task:**
1. Load the playlist data from the URL below
2. Parse it into a list of playlists, where each playlist is a list of song ID strings
3. Remove playlists with only one song
4. Load the song metadata (title, artist) into a DataFrame
5. Print: number of playlists, average playlist length, first 3 playlists

**Data URLs:**
- Playlists: `https://storage.googleapis.com/maps-premium/dataset/yes_complete/train.txt`
- Song metadata: `https://storage.googleapis.com/maps-premium/dataset/yes_complete/song_hash.txt`

**Hint:** Skip the first 2 lines of train.txt (metadata). Split each line by whitespace for song IDs.

In [ ]:
import pandas as pd
from urllib.request import urlopen

# YOUR CODE HERE

# Load playlists
# data = urlopen('https://storage.googleapis.com/maps-premium/dataset/yes_complete/train.txt')
# lines = data.read().decode('utf-8').split('\n')[2:]  # skip first 2 metadata lines
# playlists = [s.strip().split() for s in lines if len(s.strip().split()) > 1]

# Load song metadata
# songs_file = urlopen('https://storage.googleapis.com/maps-premium/dataset/yes_complete/song_hash.txt')
# songs_data = songs_file.read().decode('utf-8').split('\n')
# songs = [s.strip().split('\t') for s in songs_data if s.strip()]
# songs_df = pd.DataFrame(songs, columns=['id', 'title', 'artist'])
# songs_df = songs_df.set_index('id')

# print(f"Number of playlists: {len(playlists)}")
# print(f"Average playlist length: {sum(len(p) for p in playlists) / len(playlists):.1f} songs")
# print(f"\nFirst 3 playlists:")
# for p in playlists[:3]:
#     print(f"  {p[:5]}... ({len(p)} songs)")

## Exercise 6.2 — Train a Word2Vec Model on Playlists

**Task:** Train a Word2Vec model on the playlist data.

**Parameters to use:**
- `vector_size=32` — embedding dimension for each song
- `window=20` — consider songs up to 20 positions apart as context
- `negative=50` — use 50 negative samples per positive pair
- `min_count=1` — include all songs even if they appear once
- `workers=4` — use 4 CPU threads

**After training, print:**
- Number of songs in the model's vocabulary
- Shape of the embedding for song `"2172"`

In [ ]:
from gensim.models import Word2Vec

# YOUR CODE HERE
# song_model = Word2Vec(
#     playlists,
#     vector_size=32,
#     window=20,
#     negative=50,
#     min_count=1,
#     workers=4
# )

# print(f"Songs in vocabulary: {len(song_model.wv.key_to_index)}")
# print(f"Embedding shape for song 2172: {song_model.wv['2172'].shape}")

## Exercise 6.3 — Recommend Similar Songs

**Task:**
1. Look up what song ID `2172` is in `songs_df`
2. Find the 10 most similar songs using `song_model.wv.most_similar`
3. Write a `recommend(song_id, topn=5)` function that:
   - Prints the seed song (title + artist)
   - Prints the recommended songs with similarity scores
4. Test it on: song `2172`, and one other song ID of your choice

**Expected for song 2172 (Fade To Black — Metallica):** Similar heavy metal songs — Iron Maiden, Rush, Van Halen, Guns N' Roses, Dio.

In [ ]:
# YOUR CODE HERE

# Step 1: What is song 2172?
# print("Seed song:")
# print(songs_df.loc['2172'])

# Step 2: Find similar songs
def recommend(song_id: str, topn: int = 5) -> None:
    """Print the seed song and its most similar songs by embedding."""
    # YOUR CODE HERE
    # 1. Print seed song info from songs_df
    # 2. Call song_model.wv.most_similar(positive=song_id, topn=topn)
    # 3. Look up each similar song ID in songs_df and print with similarity score
    pass


recommend('2172')

---
## Chapter 2 Summary

You have now implemented the entire token-to-embedding pipeline from the chapter:

| Concept | What you built | Key insight |
|---|---|---|
| Tokenisation | Token IDs, per-token decoding, generation inspection | LLMs read integers, not text |
| Token count analysis | Chars-per-token across text types | Code and emojis cost more tokens |
| Tokenizer comparison | `show_tokens` visualiser, 4 tokenizers compared | Different algorithms → different splits |
| Contextualized embeddings | DeBERTa forward pass, same-word comparison | Same token ID → different vectors by context |
| Sentence embeddings | `all-mpnet-base-v2`, semantic similarity ranking | One vector captures full sentence meaning |
| GloVe word embeddings | Similar word lookup, vector arithmetic | king − man + woman ≈ queen |
| Word2Vec recommendation | Playlist-trained model, song similarity | Any sequence can be embedded with word2vec |